In [2]:
import sys
import imblearn
import xgboost

print("Python:", sys.executable)
print("imbalanced-learn:", imblearn.__version__)
print("xgboost:", xgboost.__version__)

Python: /Users/mac/Desktop/FINAL PROJECT/.venv/bin/python
imbalanced-learn: 0.14.2
xgboost: 3.4.1


In [5]:
import pandas as pd

df = pd.read_csv("/Users/mac/Desktop/FINAL PROJECT/creditcard.csv")

duplicate_mask = df.duplicated()

print("Total rows:", len(df))
print("Extra duplicate rows:", duplicate_mask.sum())

print("\nExtra duplicates by class:")
print(df.loc[duplicate_mask, "Class"].value_counts().sort_index())

clean_df = df.drop_duplicates()

print("\nAfter removing exact duplicates:")
print("Shape:", clean_df.shape)
print("Class counts:")
print(clean_df["Class"].value_counts().sort_index())

Total rows: 284807
Extra duplicate rows: 1081

Extra duplicates by class:
Class
0    1062
1      19
Name: count, dtype: int64

After removing exact duplicates:
Shape: (283726, 31)
Class counts:
Class
0    283253
1       473
Name: count, dtype: int64


SECTON 7 ORIGINAL CODE

In [ ]:
baseline_models = {
    "Dummy (always normal)": Pipeline([("clf", DummyClassifier(strategy="most_frequent"))]),
    "Logistic regression":   Pipeline([("prep", make_preprocessor()),
                                       ("clf", LogisticRegression(max_iter=1000))]),
    "Decision tree (depth 8)": Pipeline([("clf", DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE))]),
    # Naive Bayes needs categorical features: bin every column into 10 quantile bins,
    # Laplace smoothing alpha = 1 (tune alpha later with cross-validation)
    "Naive Bayes (10 bins, alpha=1)": Pipeline([
        ("bin", KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="quantile")),
        ("clf", CategoricalNB(alpha=1.0, min_categories=10))]),
}
if RUN_KNN:
    baseline_models["KNN (k=5)"] = Pipeline([("prep", make_preprocessor()),
                                             ("clf", KNeighborsClassifier(n_neighbors=5, n_jobs=-1))])

fitted = {}
for name, model in baseline_models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    fitted[name] = model
    evaluate("E1", name, "none", model, X_test, y_test)
    print(f"{name:35s} done in {time.time() - t0:5.1f}s")

display(results_table())

SECTION 8 ORIGINAL CODE

In [ ]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.under_sampling import RandomUnderSampler
    from sklearn.model_selection import StratifiedKFold, cross_validate

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rus_lr = ImbPipeline([
        ("prep", make_preprocessor()),
        ("rus",  RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE)),  # 1:1
        ("clf",  LogisticRegression(max_iter=1000)),
    ])
    cvres = cross_validate(rus_lr, X_train, y_train, cv=cv,
                           scoring={"recall": "recall", "precision": "precision",
                                    "pr_auc": "average_precision"})
    for k in ("recall", "precision", "pr_auc"):
        v = cvres["test_" + k]
        print(f"{k:10s} mean {v.mean():.4f}  std {v.std():.4f}")
except ImportError:
    print("imbalanced-learn is not installed: conda install -c conda-forge imbalanced-learn")

In [1]:
%pip install imbalanced-learn xgboost


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
